## Домашнее задание №6

В данном задании мы попробуем решить задачу классификации текстов, при этом попробуем воспользоваться информативными векторными описаниями слов, о которых рассказывалось на лекции.

Идея решения данной задачи заключается в следующем.
Мы попробуем построить бинарный классификатор, основанный на нейронных сетях и логистической регрессии. Ему на вход подаётся некоторый вектор признаков. На выходе - значение предсказания построенной модели логистической регрессии (одно число).

В качестве признаков предлагается использовать векторное представление текста. Это может быть либо классическая модель мешка слов, либо модель мешка эмбеддингов. Соответственно, модель должна по заданному векторному представлению каждого текста предсказать вероятности соответствующих классов.
Т.е. нам нужно самостоятельно реализовать аналог sklearn.linear_model.LogisticRegression с помощью библиотеки PyTorch.

In [ ]:
# Не меняйте следующий блок кода! Он потребуется для дальнейшей работы
# __________start of block__________
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator
from sklearn.linear_model import LogisticRegression
from sklearn import naive_bayes
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score

import torch
from torch import nn
from torch.nn import functional as F
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau



from IPython import display
import matplotlib.pyplot as plt
%matplotlib inline


out_dict = dict()
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# __________end of block__________

### Предобработка текста и токенизация



Для начала скачаем исходные данные:

In [ ]:
df = pd.read_csv( # Считываем исходный набор данных, разделитель - символ табуляции, заголовок отсутствует
    'https://github.com/clairett/pytorch-sentiment-classification/raw/master/data/SST2/train.tsv',
    delimiter='\t',
    header=None
)
df.head(5) # Посмотрим на первые 5 строк в считанном наборе данных

Исходный набор данных представляет собой список отзывов на различные фильмы, а также тональность соответствующего отзыва (положительный или отрицательный). Вам нужно решить задачу сентимент-анализа, т.е. построить бинарный классификатор, который позволит по известному отзыву сказать, положительный он или отрицательный.

Предобработка входных данных достаточно проста и подробно описывается в комментариях к коду.

Библиотека [`nltk`](https://www.nltk.org) широко используется при обработке текстов. По ссылке выше можно найти ее развернутое описание и документацию.

In [ ]:
# Не изменяйте следующий блок кода! Он нужен для корректной предобработки входных данных

# __________start of block__________

texts_train = df[0].values[:5000] # В качестве обучающей выборки выбираем первые 5000 предложений
y_train = df[1].values[:5000] # Каждому предложению соответствует некоторая метка класса - целое число
texts_test = df[0].values[5000:] # В качестве тестовой выборки используем все оставшиеся предложения
y_test = df[1].values[5000:]

from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer() # В качестве токенов будем использовать отдельные слова и знаки препинания

# В качестве предобработки будем приводить текст к нижнему регистру.
# Предобработанный текст будем представлять в виде выделенных токенов, разделённых пробелом
preprocess = lambda text: ' '.join(tokenizer.tokenize(text.lower()))

text = 'How to be a grown-up at work: replace "I don\'t want to do that" with "Ok, great!".'
print("before:", text,)
print("after:", preprocess(text),) # Посмотрим, как работает предобработка для заданной строки text

texts_train = [preprocess(text) for text in texts_train] # Получаем предобработанное представление для тренировочной выборки
texts_test = [preprocess(text) for text in texts_test] # Аналогично получаем предобработанное представление для тестовой выборки

# Выполняем небольшие проверки того, насколько корректно были обработаны тренировочная и тестовая выборки
assert texts_train[5] ==  'campanella gets the tone just right funny in the middle of sad in the middle of hopeful'
assert texts_test[74] == 'poetry in motion captured on film'
assert len(texts_test) == len(y_test)
# __________end of block__________

Следующие функции помогут вам с визуализацией процесса обучения сети.

In [ ]:
# Не изменяйте блок кода ниже!

# __________start of block__________
def plot_train_process(train_loss, val_loss, train_accuracy, val_accuracy, title_suffix=''):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    axes[0].set_title(' '.join(['Loss', title_suffix]))
    axes[0].plot(train_loss, label='train')
    axes[0].plot(val_loss, label='validation')
    axes[0].legend()

    axes[1].set_title(' '.join(['Validation accuracy', title_suffix]))
    axes[1].plot(train_accuracy, label='train')
    axes[1].plot(val_accuracy, label='validation')
    axes[1].legend()
    plt.show()

def visualize_and_save_results(model, model_name, X_train, X_test, y_train, y_test, out_dict):
    for data_name, X, y, model in [
    ('train', X_train, y_train, model),
    ('test', X_test, y_test, model)
    ]:
        if isinstance(model, BaseEstimator):
            proba = model.predict_proba(X)[:, 1]
        elif isinstance(model, nn.Module):
            proba = model(X).detach().cpu().numpy()[:, 1]
        else:
            raise ValueError('Unrecognized model type')

        auc = roc_auc_score(y, proba)

        out_dict['{}_{}'.format(model_name, data_name)] = auc
        plt.plot(*roc_curve(y, proba)[:2], label='%s AUC=%.4f' % (data_name, auc))

    plt.plot([0, 1], [0, 1], '--', color='black',)
    plt.legend(fontsize='large')
    plt.title(model_name)
    plt.grid()
    return out_dict
# __________end of block__________

### Задача №1. Мешок слов.

На лекции рассматривались подходы, позволяющие представить отдельные токены в виде векторов. Однако, в реальной жизни мы работаем с полноценными текстами.

Как известно из лекции, **текст** -- последовательность токенов, длина которой **не фиксирована**. Соответственно, если каждый текст представлять матрицей, где каждый столбец будет являться векторным описанием очередного токена, мы будем получать для разных текстов матрицы разного размера. Как мы знаем, такие матрицы непригодны для передачи в нейронные сети, поскольку размер входных данных сети - всегда фиксирован. Поэтому нужно научиться представлять любой текст некоторым тензором (вектором или матрицей) фиксированного размера.

Самый простой способ -- воспользоваться классическим подходом к векторизации текстов: мешком слов. На лекции упоминалось, что в таком подходе каждый токен может быть закодирован разреженным вектором, размер которого будет равен размеру словаря. Соответственно, чтобы получить векторное представление всего текста, достаточно сложить векторные представления всех входящих в него токенов.

Первое, что мы рассмотрим - One-hot кодирование токенов. В этом случае мы приходим к тому, что текст будет представлен вектором, длина которого равна размеру словаря, N-ым значением в этом векторе будет количество употребления слова с индексом N в исходном тексте.

Для реализации такого подхода вы можете как воспользоваться `CountVectorizer` из `sklearn`, так и самостоятельно реализованный вариант. Обращаем ваше внимание, в данной задаче используется лишь `k` наиболее часто встречаемых слов из обучающей части выборки.

In [ ]:
# Не изменяйте блок кода ниже!

# __________start of block__________

# Отбираем только k наиболее популярных слов в текстах

k = min(10000, len(set(' '.join(texts_train).split()))) # Если в словаре меньше 10000 слов, то берём все слова, в противном случае выберем 10000 самых популярных

# Построим словарь всех уникальных слов в обучающей выборке,
# оставив только k наиболее популярных слов.

counts = Counter(' '.join(texts_train).split())
bow_vocabulary = [key for key, val in counts.most_common(k)]


def text_to_bow(text):
    """ Функция, позволяющая превратить входную строку в векторное представление на основании модели мешка слов. """
    sent_vec = np.zeros(len(bow_vocabulary))
    counts = Counter(text.split())
    for i, token in enumerate(bow_vocabulary):
        if token in counts:
            sent_vec[i] = counts[token]
    return np.array(sent_vec, 'float32')

X_train_bow = np.stack(list(map(text_to_bow, texts_train)))
X_test_bow = np.stack(list(map(text_to_bow, texts_test)))

# Небольшие проверки - они нужны, если Вы захотите реализовать собственную модель мешка слов.
k_max = len(set(' '.join(texts_train).split()))
assert X_train_bow.shape == (len(texts_train), min(k, k_max))
assert X_test_bow.shape == (len(texts_test), min(k, k_max))
assert np.all(X_train_bow[5:10].sum(-1) == np.array([len(s.split()) for s in  texts_train[5:10]]))
assert len(bow_vocabulary) <= min(k, k_max)
assert X_train_bow[65, bow_vocabulary.index('!')] == texts_train[65].split().count('!')


# Строим модель логистической регрессии для полученных векторных представлений текстов
bow_model = LogisticRegression(max_iter=1500).fit(X_train_bow, y_train)

out_dict = visualize_and_save_results(bow_model, 'bow_log_reg_sklearn', X_train_bow, X_test_bow, y_train, y_test, out_dict)
# __________end of block__________

Результаты неплохие, но явно видно переобучение. Этот вывод можно сделать судя по значительному превосходству качества (AUC ROC) на train выборке относительно test. Более того, на обучающей выборке качество стремится к единице, в то время как на отложенной – значительно ниже, т.е. модель уловила множество зависимостей, свойственных лишь обучающей выборке. Базово проблема переобучения рассматривалась в курсе Машинного обучения. Более подробно она еще не раз встретится при дальнейшем прохождении курса.

В данной задаче с переобучением мы разберемся в дальнейшем. Сейчас же реализуйте решение на основе логистической регрессии, но уже используя PyTorch. В результате вам должна быть доступна обученная модель, предсказывающая вероятности для двух классов. Качество на тестовой выборке должно не уступать логистической регрессии.

Таким образом, нужно реализовать однослойную линейную сеть с использованием softmax в качестве функции активации и количеством выходов, равному количеству классов.

In [ ]:
model = nn.Linear(len(bow_vocabulary), 2).to(device)

Не забывайте о функциях потерь: `nn.CrossEntropyLoss` объединяет в себе `LogSoftMax` и `NLLLoss`. Также не забывайте о необходимости перенести тензоры на используемый `device`.

In [ ]:
loss_function = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
X_train_bow_torch = torch.tensor(X_train_bow, dtype=torch.float32).to(device)
X_test_bow_torch = torch.tensor(X_test_bow, dtype=torch.float32).to(device)
y_train_torch = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_torch = torch.tensor(y_test, dtype=torch.long).to(device)

Функция ниже поможет с обучением модели. Часть кода необходимо реализовать самостоятельно.

In [ ]:
def train_model(
    model,
    opt,
    X_train_torch,
    y_train_torch,
    X_val_torch,
    y_val_torch,
    n_iterations=500,
    batch_size=32,
    show_plots=True,
    eval_every=50
):
    train_loss_history = []
    train_acc_history = []
    val_loss_history = []
    val_acc_history = []

    local_train_loss_history = []
    local_train_acc_history = []
    for i in range(n_iterations):

        # Получаем случайный батч размера batch_size для проведения обучения
        ix = np.random.randint(0, len(X_train_torch), batch_size)
        x_batch = X_train_torch[ix]
        y_batch = y_train_torch[ix]

        # Предсказываем отклик (log-probabilities или logits)
        y_predicted = model(x_batch)
        loss = loss_function(y_predicted, y_batch)
        loss.backward()
        opt.step()
        opt.zero_grad()


        local_train_loss_history.append(loss.item())
        local_train_acc_history.append(
            accuracy_score(
                y_batch.to('cpu').detach().numpy(),
                y_predicted.to('cpu').detach().numpy().argmax(axis=1)
            )
        )

        if i % eval_every == 0:
            train_loss_history.append(np.mean(local_train_loss_history))
            train_acc_history.append(np.mean(local_train_acc_history))
            local_train_loss_history, local_train_acc_history = [], []

            predictions_val = model(X_val_torch)
            val_loss_history.append(loss_function(predictions_val, y_val_torch).to('cpu').detach().item())

            acc_score_val = accuracy_score(y_val_torch.cpu().numpy(), predictions_val.to('cpu').detach().numpy().argmax(axis=1))
            val_acc_history.append(acc_score_val)

            if show_plots:
                display.clear_output(wait=True)
                plot_train_process(train_loss_history, val_loss_history, train_acc_history, val_acc_history)
    return model

In [ ]:
bow_nn_model = train_model(model, opt, X_train_bow_torch, y_train_torch, X_test_bow_torch, y_test_torch, n_iterations=3000)

In [ ]:
# Не изменяйте блок кода ниже!
# __________start of block__________
out_dict = visualize_and_save_results(bow_nn_model, 'bow_nn_torch', X_train_bow_torch, X_test_bow_torch, y_train, y_test, out_dict)

assert out_dict['bow_log_reg_sklearn_test'] - out_dict['bow_nn_torch_test'] < 0.01, 'AUC ROC on test data should be close to the sklearn implementation'
# __________end of block__________

А теперь повторите процедуру обучения выше, но для различных значений `k` – размера словаря. В список results сохраните `AUC ROC` на тестовой части выборки для модели, обученной со словарем размера `k`.

In [ ]:
vocab_sizes_list = np.arange(100, 5800, 700)
results = []

for k in vocab_sizes_list:
        # Строим словарь из k самых частотных слов
    counts = Counter(' '.join(texts_train).split())
    bow_vocabulary_k = [key for key, val in counts.most_common(k)]
    
    # Функция преобразования текста в BoV вектор
    def text_to_bow_k(text):
        vec = np.zeros(len(bow_vocabulary_k))
        cnt = Counter(text.split())
        for i, token in enumerate(bow_vocabulary_k):
            if token in cnt:
                vec[i] = cnt[token]
        return vec
    
    X_train_k = np.stack([text_to_bow_k(t) for t in texts_train])
    X_test_k = np.stack([text_to_bow_k(t) for t in texts_test])
    
    # Переводим в тензоры
    X_train_k_torch = torch.tensor(X_train_k, dtype=torch.float32).to(device)
    X_test_k_torch = torch.tensor(X_test_k, dtype=torch.float32).to(device)
    y_train_torch = torch.tensor(y_train, dtype=torch.long).to(device)
    y_test_torch = torch.tensor(y_test, dtype=torch.long).to(device)
    
    # Создаём модель и оптимизатор
    model_k = nn.Linear(len(bow_vocabulary_k), 2).to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt_k = torch.optim.Adam(model_k.parameters(), lr=1e-3)
    
    # Обучаем (без отображения графиков)
    model_k = train_model(model_k, opt_k, X_train_k_torch, y_train_torch,
                          X_test_k_torch, y_test_torch, n_iterations=3000, show_plots=False)
    
    # Предсказываем вероятности для теста
    with torch.no_grad():
        logits = model_k(X_test_k_torch)
        predicted_probas_on_test_for_k_sized_dict = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    
    assert predicted_probas_on_test_for_k_sized_dict is not None
    auc = roc_auc_score(y_test, predicted_probas_on_test_for_k_sized_dict)
    results.append(auc)

In [ ]:
# Не меняйте блок кода ниже!

# __________start of block__________
assert len(results) == len(vocab_sizes_list), 'Check the code above'
assert min(results) >= 0.65, 'Seems like the model is not trained well enough'
assert results[-1] > 0.84, 'Best AUC ROC should not be lower than 0.84'

plt.plot(vocab_sizes_list, results)
plt.xlabel('num of tokens')
plt.ylabel('AUC')
plt.grid()

out_dict['bow_k_vary'] = results
# __________end of block__________

### Задача №2: Использование TF-iDF признаков.

Для векторизации текстов также можно воспользоваться TF-iDF. Это позволяет исключить из рассмотрения многие слова, не оказывающие значимого влияния при оценке непохожести текстов.

В лекции уже упоминалось про то, как векторизовать отдельные токены с помощью этого подхода. Соответственно, векторным представлением текста будет являться вектор размера словаря, где в компоненте с номером N будет указано значение TF-IDF в данном тексте для токена с индексом N.

Подробнее про TF-iDF можно почитать, например, [здесь](https://habr.com/en/companies/otus/articles/755772/).

Ваша задача: векторизовать тексты используя TF-iDF (или `TfidfVectorizer` из `sklearn`, или реализовав его самостоятельно) и построить классификатор с помощью PyTorch, аналогичный задаче №1.

Затем также оцените качество классификации по AUC ROC для различных размеров словаря.

Качество классификации должно быть не ниже 0.86 AUC ROC.

In [ ]:
# Вставьте свой код для векторизации текстов с помощью TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(vocabulary=bow_vocabulary)  # используем тот же словарь, что и в BoW
X_train_tfidf = tfidf_vectorizer.fit_transform(texts_train).toarray().astype('float32')
X_test_tfidf = tfidf_vectorizer.transform(texts_test).toarray().astype('float32')

In [ ]:
model = nn.Linear(X_train_tfidf.shape[1], 2).to(device)
loss_function = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
X_train_tfidf_torch = torch.tensor(X_train_tfidf, dtype=torch.float32).to(device)
X_test_tfidf_torch = torch.tensor(X_test_tfidf, dtype=torch.float32).to(device)
model_tf_idf = train_model(model, opt, X_train_tfidf_torch, y_train_torch, X_test_tfidf_torch, y_test_torch, n_iterations=3000)

In [ ]:
# Не меняйте блок кода ниже!

# __________start of block__________
out_dict = visualize_and_save_results(model_tf_idf, 'tf_idf_nn_torch', X_train_tfidf_torch, X_test_tfidf_torch, y_train, y_test, out_dict)

assert out_dict['tf_idf_nn_torch_test'] >= out_dict['bow_nn_torch_test'], 'AUC ROC on test data should be better or close to BoW for TF-iDF features'
# __________end of block__________

Аналогично задаче №1 повторите процедуру обучения для различных значений `k` – размера словаря и сохраните `AUC ROC` на тестовой части выборки в список `results`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vocab_sizes_list = np.arange(100, 5800, 700)
results = []

for k in vocab_sizes_list:
    vectorizer = TfidfVectorizer(max_features=k, token_pattern=r'\S+')
    X_train_k = vectorizer.fit_transform(texts_train).toarray().astype('float32')
    X_test_k = vectorizer.transform(texts_test).toarray().astype('float32')
    
    X_train_k_torch = torch.tensor(X_train_k, dtype=torch.float32).to(device)
    X_test_k_torch = torch.tensor(X_test_k, dtype=torch.float32).to(device)
    y_train_torch = torch.tensor(y_train, dtype=torch.long).to(device)
    y_test_torch = torch.tensor(y_test, dtype=torch.long).to(device)
    
    model_k = nn.Linear(X_train_k.shape[1], 2).to(device)
    loss_fn = nn.CrossEntropyLoss()
    opt_k = torch.optim.Adam(model_k.parameters(), lr=1e-3)
    model_k = train_model(model_k, opt_k, X_train_k_torch, y_train_torch,
                          X_test_k_torch, y_test_torch, n_iterations=3000, show_plots=False)
    
    with torch.no_grad():
        logits = model_k(X_test_k_torch)
        predicted_probas_on_test_for_k_sized_dict = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
    
    auc = roc_auc_score(y_test, predicted_probas_on_test_for_k_sized_dict)
    results.append(auc)

In [ ]:
# Не меняйте блок кода ниже!

# __________start of block__________
assert len(results) == len(vocab_sizes_list), 'Check the code above'
assert min(results) >= 0.65, 'Seems like the model is not trained well enough'
assert results[-1] > 0.85, 'Best AUC ROC for TF-iDF should not be lower than 0.84'

plt.plot(vocab_sizes_list, results)
plt.xlabel('num of tokens')
plt.ylabel('AUC')
plt.grid()

out_dict['tf_idf_k_vary'] = results
# __________end of block__________

### Задача №3: Сравнение с Наивным Байесовским классификатором.

Классические модели все еще способны показать хороший результат во многих задачах. Обучите наивный байесовский классификатор на текстах, векторизованных с помощью BoW и TF-iDF и сравните результаты с моделями выше.

*Комментарий: обращаем ваше внимание, необходимо выбрать подходящее к данной задаче априорное распределение для признаков, т.е. выбрать верную версию классификатора из `sklearn`: `GaussianNB`, `MultinomialNB`, `ComplementNB`, `BernoulliNB`, `CategoricalNB`*

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
clf_nb_bow = MultinomialNB().fit(X_train_bow, y_train)

# do not change the code in the block below
# __________start of block__________
out_dict = visualize_and_save_results(clf_nb_bow, 'bow_nb_sklearn', X_train_bow, X_test_bow, y_train, y_test, out_dict)
# __________end of block__________

In [ ]:
clf_nb_tfidf = MultinomialNB().fit(X_train_tfidf, y_train)

# do not change the code in the block below
# __________start of block__________
out_dict = visualize_and_save_results(clf_nb_tfidf, 'tf_idf_nb_sklearn', X_train_tfidf, X_test_tfidf, y_train, y_test, out_dict)
# __________end of block__________

In [ ]:
# do not change the code in the block below
# __________start of block__________
assert out_dict['tf_idf_nb_sklearn_test'] > out_dict['bow_nb_sklearn_test'],' TF-iDF results should be better'
assert out_dict['tf_idf_nb_sklearn_test'] > 0.86, 'TF-iDF Naive Bayes score should be above 0.86'
# __________end of block__________

### Задача №4: Использование предобученных эмбеддингов

Наконец, воспользуемся предобученными эмбеддингами из библиотеки `gensim`. В нем доступно несколько эмбеддингов, предобученных на различных корпусах текстов. Полный список можно найти [здесь](https://radimrehurek.com/gensim/models/word2vec.html#pretrained-models). Напоминаем, что лучше использовать те эмбеддинги, которые были обучены на текстах похожей структуры.

Ваша задача: обучить модель (достаточно логистической регрессии или же двуслойной неронной сети), используя усредненный эмбеддинг для всех токенов в отзыве, добиться качества не хуже, чем с помощью BoW/TF-iDF и снизить степень переобучения (разницу между AUC ROC на обучающей и тестовой выборках).

**Обратите внимание!**

В Google Colab установлена устаревшая версия библиотеки gensim, поэтому некоторые её обновлённые возможности могут не поддерживаться.

Например, может возникать ошибка AttributeError.

В этом случае нужно раскомментировать строку ниже, выполнить данную ячейку и повторно выполнить все последующие ячейки:

In [ ]:
#!pip install --upgrade gensim

In [ ]:
import gensim.downloader as api
gensim_embedding_model = api.load("glove-twitter-200")

In [ ]:
def text_to_average_embedding(text, gensim_embedding_model):
    words = text.split()
    embeds = [gensim_embedding_model[w] for w in words if w in gensim_embedding_model]
    if embeds:
        return np.mean(embeds, axis=0)
    else:
        return np.zeros(gensim_embedding_model.vector_size)
    return embedding_for_text

In [ ]:
X_train_emb = [text_to_average_embedding(text, gensim_embedding_model) for text in texts_train]
X_test_emb = [text_to_average_embedding(text, gensim_embedding_model) for text in texts_test]

assert len(X_train_emb[0]) == gensim_embedding_model.vector_size, 'Seems like the embedding shape is wrong'

In [ ]:
X_train_emb_torch = torch.tensor(np.stack(X_train_emb), dtype=torch.float32).to(device)
X_test_emb_torch = torch.tensor(np.stack(X_test_emb), dtype=torch.float32).to(device)
y_train_torch = torch.tensor(y_train, dtype=torch.long).to(device)
y_test_torch = torch.tensor(y_test, dtype=torch.long).to(device)

In [ ]:
model = nn.Linear(X_train_emb_torch.shape[1], 2).to(device)

In [ ]:
loss_function = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model = train_model(model, opt, X_train_emb_torch, y_train_torch, X_test_emb_torch, y_test_torch, n_iterations=7000)

In [ ]:
# do not change the code in the block below
# __________start of block__________

out_dict = visualize_and_save_results(model, 'emb_nn_torch', X_train_emb_torch, X_test_emb_torch, y_train, y_test, out_dict)
assert out_dict['emb_nn_torch_test'] > 0.87, 'AUC ROC on test data should be better than 0.87'
assert out_dict['emb_nn_torch_train'] - out_dict['emb_nn_torch_test'] < 0.1, 'AUC ROC on test and train data should not be different more than by 0.1'
# __________end of block__________

### Сдача задания
Запустите код ниже для генерации посылки и сдайте на проверку файл `submission_dict_hw06.npy`.

In [ ]:
# Не меняйте блок кода ниже!

# __________start of block__________

np.save('submission_dict_hw06.npy', out_dict, allow_pickle=True)
print('File saved to `submission_dict_hw06.npy`')
# __________end of block__________

### Описание возможных ошибок



Ниже представлено краткое описание тестов, которые запускаются для Вашего решения, и причин, по которым они не могут быть пройдены:

1. `test_tfidf_on_test` - значение ROC-AUC на тестовой выборке при применении TF-IDF к тестовому набору данных недостаточно высоко (см. условие).

2. `test_tfidf_nb_first` - возможно, выбрано некорректное распределение для Наивного Байесовского классификатора, качество для BoW превосходит TF-IDF.

3. `test_tfidf_nb_second` - возможно, выбрано некорректное распределение для Наивного Байесовского классификатора: качество на тестовой выборке ниже необходимого (см. условие).

4. `test_embedding_on_test` - недостаточно высокое качество для модели на основе эмбеддингов.

5. `test_embedding_overfit` - модель, использующая эмбеддинги, переобучается с большой долей вероятности.